# LoRa network simulation on the campus: PDR, latency, RSSI and SNR

This notebook turns the campus ray-tracing scene into a **packet-level LoRa network experiment**. The variables are the number of nodes, their positions and their LoRa settings (SF, bandwidth, coding rate, payload, TX power, channel, traffic and MAC options). The outputs are packet delivery ratio (PDR), end-to-end latency, RSSI/SNR of received packets, retransmissions, collisions, channel utilisation and an energy estimate.

```text
node positions ──► Sionna RT link gains ──► [calibration from your RSSI/SNR logs] ──► discrete-event packet sim ──► metrics
   (sampled or       (N×N, cached)                (offsets, spreads, noise,           (airtime, collisions/capture,
    your real ones)                                 PER curve)                          half-duplex, ACK/retries, duty cycle)
```

The simulator lives in [`lora_netsim.py`](lora_netsim.py) and is covered by NumPy-only tests in [`test_lora_netsim.py`](test_lora_netsim.py). Ray tracing runs once per set of positions and is cached, so changing LoRa or traffic settings and rerunning the simulation takes seconds.

**VS Code:** select the kernel, then Run All. CPU ray tracing is the default; the first gain computation takes a few minutes and later runs load the cache.

Every default below is an explicit assumption: material proxies, the PER waterfall, interference thresholds, fading and energy currents. Section 4 replaces the propagation and noise terms with values fitted to your measurements.

In [ ]:
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from matplotlib.collections import LineCollection

import mitsuba as mi
mi.set_variant("llvm_ad_mono_polarized")  # CPU; "cuda_ad_mono_polarized" on an NVIDIA GPU. Restart kernel to change.

def find_repo_root(start=Path.cwd()):
    for parent in (start, *start.parents):
        for candidate in (parent, parent / "sionna-lora-ext"):
            if (candidate / "assets/sionna/scene_923MHz.xml").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the nvidia or sionna-lora-ext workspace.")

REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "tutorials" / "rt-campus"))
import lora_netsim as ln

print("Repository:", REPO_ROOT, "| Mitsuba variant:", mi.variant())

## 1. Experiment configuration

Edit this cell, then rerun from here. Key choices:

- **Topology.** `"star"`: every end node sends to the gateway(s), like LoRaWAN uplinks. `"peer"`: each node sends to one other node (`PEER_DEST`), like the six-node pilot in notebook 1. Peer receivers listen continuously on their own SF and channel.
- **Positions.** Nodes are sampled with a seed inside the audited public regions. To use your real deployment, set `MANUAL_POSITIONS` to `(name, floor, x, y)` tuples in scene coordinates.
- **LoRa settings.** `LORA` is the default for every node. `SF_PER_NODE` / `POWER_PER_NODE` override individual nodes, for example `{0: 9, 4: 12}`.
- **MAC.** `confirmed=True` adds ACKs (sent 1 s after the uplink on the same SF) and up to `max_retransmissions` retries with random backoff. `duty_cycle=0.01` enforces a 1% off-time. Check the regional rules that apply to your deployment.

In [ ]:
SEED = 42
TOPOLOGY = "star"                      # "star" | "peer"
N_END_NODES = 8
GATEWAYS = [("GW1", "F1", -29.0, 41.0)]  # (name, floor, x, y); ground-floor atrium. Ignored for "peer".
PEER_DEST = "next"                     # "peer": node k sends to node (k+1) mod N
MANUAL_POSITIONS = None                # e.g. [("N1", "F1", -40.0, 14.0), ("N2", "F2", -48.0, 27.0), ...]
ANTENNA_HEIGHT_M = 1.3                 # Above the finished floor
MIN_SPACING_M = 3.0                    # Horizontal, same floor (incl. gateway)

LORA = ln.LoRaConfig(sf=7, bw_hz=125e3, cr=1, payload_bytes=16, preamble_symbols=8, tx_power_dbm=14.0)
SF_PER_NODE = {}                       # {node index: SF}
POWER_PER_NODE = {}                    # {node index: dBm}
TRAFFIC_PERIOD_S = 60.0                # Mean interval between messages per node
TRAFFIC = "poisson"                    # "poisson" | "periodic"

SIM = ln.SimConfig(duration_s=3600.0, confirmed=True, max_retransmissions=3, ack_delay_s=1.0,
                   duty_cycle=None, seed=SEED)
RADIO = ln.RadioModel(noise_figure_db=6.0, cable_loss_db=1.0, fading_std_db=2.0)

# Ray tracing (identical for every LoRa setting at the same carrier and positions).
FREQUENCY_HZ = 923e6
ANTENNA_PEAK_DBI = 1.96                # RAKARJ10 at 923 MHz (see README); hw_dipole surrogate scaled to it
RT_KW = dict(bandwidth_hz=LORA.bw_hz, antenna_correction_db=ANTENNA_PEAK_DBI - ln.HW_DIPOLE_PEAK_DBI,
             max_depth=8, samples_per_src=1_000_000, diffraction=True,
             local_avg_radius_m=0.15, local_avg_points=5, seed=SEED)
# CPU cost at these defaults: ~1 min per site, ~5 GB RAM peak. Lower samples_per_src for a quick look.
CACHE_DIR = REPO_ROOT / "tutorials" / "rt-campus" / ".rt_cache"

# Your measurements (section 4). See measurements_template.csv for the columns.
MEASUREMENTS_CSV = None                # e.g. REPO_ROOT / "data" / "campus_rssi_snr.csv"
RSSI_KIND = "total"                    # "total": packet RSSI = signal + noise (SX126x RssiPkt); "signal": SignalRssiPkt
LINK_CLASSES = "floor"                 # "floor": separate offsets for same-floor / cross-floor links; None: one offset

# Sweep (section 7).
SWEEP_NODE_COUNTS = [2, 4, 8, 12]      # Pool size = max; the six audited regions hold ~12 nodes at 3 m
SWEEP_SFS = [7, 9, 12]
SWEEP_SEEDS = [0, 1, 2]
SWEEP_PERIOD_S = 30.0

assert TOPOLOGY in ("star", "peer") and max(SWEEP_NODE_COUNTS) >= N_END_NODES
assert N_END_NODES >= (2 if TOPOLOGY == "peer" else 1) and min(SWEEP_NODE_COUNTS) >= (2 if TOPOLOGY == "peer" else 1)

## 2. Load the scene and place the nodes

Sampled nodes must stand on the right floor sheet with clearance from nearby geometry, and keep `MIN_SPACING_M` from each other and from the gateway on the same floor. The six audited public regions (`ln.CAMPUS_REGIONS`) are small. For more nodes, append further regions after checking them on the floor plans, or give your real coordinates. One **pool** of `max(SWEEP_NODE_COUNTS)` nodes is placed and traced once. The main experiment uses its first `N_END_NODES`, and the sweep uses growing prefixes, so the node count changes without retracing.

In [ ]:
scene = ln.load_campus_scene(REPO_ROOT / "assets/sionna/scene_923MHz.xml", frequency_hz=FREQUENCY_HZ)
floors = ln.CAMPUS_FLOORS

def make_site(name, floor, x, y):
    return {"name": name, "floor": floor, "region": "manual",
            "position": np.array([x, y, floors[floor]["surface_z"] + ANTENNA_HEIGHT_M])}

gateway_sites = [make_site(*g) for g in GATEWAYS] if TOPOLOGY == "star" else []
for site in gateway_sites:
    assert ln.geometry_clear(scene, [site["position"]], site["floor"])[0], f"{site['name']} lacks floor/clearance"

if MANUAL_POSITIONS is None:
    pool_size = max(SWEEP_NODE_COUNTS)
    counts = ln.round_robin_counts(pool_size, len(ln.CAMPUS_REGIONS))
    pool = ln.sample_positions(scene, counts, seed=SEED, antenna_height_m=ANTENNA_HEIGHT_M,
                               min_spacing_m=MIN_SPACING_M, existing=gateway_sites)
    # Interleave regions so every prefix of the pool spreads over the building.
    order = np.argsort([k for c in counts for k in range(c)], kind="stable")
    pool = [pool[k] for k in order]
    for k, site in enumerate(pool):
        site["name"] = f"N{k + 1}"
else:
    pool = [make_site(*m) for m in MANUAL_POSITIONS]
    SWEEP_NODE_COUNTS = [k for k in SWEEP_NODE_COUNTS if k <= len(pool)]
    N_END_NODES = min(N_END_NODES, len(pool))

sites = gateway_sites + pool                  # Gateways first, then the end-node pool
positions = np.array([s["position"] for s in sites])
site_floors = [s["floor"] for s in sites]
G = len(gateway_sites)
distance = np.linalg.norm(positions[:, None] - positions[None], axis=-1)
assert np.all(distance[~np.eye(len(sites), dtype=bool)] >= 0.5)

rows = ["| Site | Role | Floor | Region | x | y | z |", "|---|---|---|---|---:|---:|---:|"]
for k, s in enumerate(sites):
    role = "gateway" if k < G else ("end node" if k - G < N_END_NODES else "sweep only")
    rows.append(f"| {s['name']} | {role} | {s['floor']} | {s['region']} | " + " | ".join(f"{v:.2f}" for v in s["position"]) + " |")
display(Markdown("\n".join(rows)))

In [ ]:
outlines = {f: ln.floor_outline_segments(scene, floors[f]["surface_z"] + ANTENNA_HEIGHT_M) for f in floors}

def draw_floor(ax, floor, values=None, label=None, cmap="RdYlGn", vmin=0, vmax=1):
    ax.add_collection(LineCollection(outlines[floor], colors="#667085", linewidths=0.4, zorder=1))
    for r in ln.CAMPUS_REGIONS:
        if r["floor"] == floor:
            x0, x1, y0, y1 = r["bounds"]
            ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor="#00a68a", alpha=0.10, zorder=0))
    mappable = None
    for k, s in enumerate(sites[:G + N_END_NODES]):
        if s["floor"] != floor:
            continue
        x, y = s["position"][:2]
        if k < G:
            ax.scatter(x, y, marker="*", s=260, c="#0072b2", edgecolor="white", zorder=5)
        elif values is None:
            ax.scatter(x, y, s=70, c="#e6550d", edgecolor="white", zorder=5)
        else:
            mappable = ax.scatter(x, y, s=110, c=[values[k - G]], cmap=cmap, vmin=vmin, vmax=vmax,
                                  edgecolor="black", zorder=5)
        ax.annotate(s["name"], (x, y), xytext=(4, 5), textcoords="offset points", fontsize=8, weight="bold", zorder=6)
    ax.set(xlim=(-70, 6), ylim=(-11, 64), aspect="equal", xlabel="x [m]", ylabel="y [m]", title=floor)
    ax.grid(alpha=0.12)
    return mappable

fig, axes = plt.subplots(1, 2, figsize=(14, 7), layout="constrained")
for ax, floor in zip(axes, floors):
    draw_floor(ax, floor)
fig.suptitle(f"{N_END_NODES} end nodes (orange){' and gateway (blue star)' if G else ''}; green = allowed regions")
plt.show()

## 3. Ray-traced link gains

Every site is traced as a transmitter to all the others. The gain includes the antenna patterns: an upright half-wave dipole scaled to `ANTENNA_PEAK_DBI`. Linear power is averaged over the LoRa bandwidth and over 5 points within 15 cm of each receiver. That smooths sub-wavelength nulls (λ ≈ 32.5 cm) that a real node would not sit in exactly, and it matches how measurements are compared in section 4.

Physical links are reciprocal, so the forward/reverse difference measures ray-sampling convergence. The two directions are then averaged. Raise `samples_per_src` / `max_depth` if many pairs disagree by more than 3 dB.

In [ ]:
links = ln.gain_matrix_db(scene, positions, cache_dir=CACHE_DIR,
                          cache_tag=f"scene_923MHz|{FREQUENCY_HZ}|{ANTENNA_HEIGHT_M}", **RT_KW)
gain_rt = links["gain_db"]
offdiag = ~np.eye(len(sites), dtype=bool)
err = links["reciprocity_error_db"][offdiag]
print(f"Forward/reverse difference: median {np.median(err[np.isfinite(err)]):.2f} dB, "
      f"{np.mean(err > 3):.0%} of directed links > 3 dB")
print("No-path links:", int(np.isneginf(gain_rt[offdiag]).sum()), "| invalid (NaN) links:", int(np.isnan(gain_rt[offdiag]).sum()))
if np.isnan(gain_rt[offdiag]).any():
    raise ValueError("Solver returned NaN in both directions for some links; rerun with diffraction=False or move those sites.")

mean_rss = LORA.tx_power_dbm - 2 * RADIO.cable_loss_db + gain_rt
fig, ax = plt.subplots(figsize=(9, 7.5), layout="constrained")
shown = slice(0, G + N_END_NODES)
im = ax.imshow(np.ma.masked_invalid(mean_rss[shown, shown]), cmap="viridis", vmin=-140, vmax=-30)
names = [s["name"] for s in sites[shown]]
ax.set(xticks=range(len(names)), yticks=range(len(names)), xticklabels=names, yticklabels=names,
       xlabel="Receiver", ylabel="Transmitter", title=f"Mean RSS at {LORA.tx_power_dbm:g} dBm before calibration [dBm]")
for i in range(len(names)):
    for j in range(len(names)):
        if i != j:
            ax.text(j, i, f"{mean_rss[i, j]:.0f}", ha="center", va="center", fontsize=7, color="white")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

## 4. Calibrate propagation with your RSSI/SNR measurements

Skip this section (leave `MEASUREMENTS_CSV = None`) until you have a log in the [template format](measurements_template.csv): one row per packet, transmitter and receiver antenna positions in **scene coordinates** (metres, the same frame as the table above), and the reported `rssi_dbm` / `snr_db`. Optional columns: `received` (0 for lost packets, which can have empty RSSI/SNR), `tx_power_dbm`, `sf`, `link_class`.

What is fitted, and why:

1. **Signal power from the report.** Packet RSSI includes noise, so the signal is `RSSI + SNR − 10·log10(1 + 10^(SNR/10))`. This matters at negative SNR, where RSSI sits on the noise floor.
2. **Offsets** (measured − predicted) per link class (same floor / cross floor). These absorb unknown cable, antenna-efficiency, body and material errors. Each location pair counts once, so long logs at one spot do not dominate.
3. **Shadowing σ:** the location-to-location scatter left after the offsets, applied as a static random term per link.
4. **Fading σ:** the packet-to-packet spread at a fixed location pair (people moving, doors), applied per packet.
5. **Noise figure:** taken from the implied noise floor `RSSI − 10·log10(1 + 10^(SNR/10))`, which includes local interference.
6. **PER curve** (only if `received` contains lost packets): reception versus predicted SNR, replacing the assumed waterfall.

The leave-one-pair-out RMSE says whether calibration actually predicts **unseen** locations better than the raw ray tracer. Report it with your results. Received-only logs under-represent weak links (lost packets have no RSSI), so near the sensitivity limit the offsets are optimistic. Logging lost packets via sequence numbers fixes this.

In [ ]:
gain_used, radio_used, calibration = gain_rt, RADIO, None
link_class = ln.floor_class_matrix(site_floors) if LINK_CLASSES == "floor" else None

if MEASUREMENTS_CSV is not None:
    meas = ln.load_measurements(MEASUREMENTS_CSV)
    if LINK_CLASSES == "floor" and np.all(meas["link_class"] == "all"):
        # Infer same/cross floor from antenna heights when the log does not label it.
        mid = (floors["F1"]["surface_z"] + floors["F2"]["support_z"]) / 2
        meas["link_class"] = np.where((meas["tx_position"][:, 2] > mid) == (meas["rx_position"][:, 2] > mid),
                                      "same_floor", "cross_floor")
    predicted = ln.pair_gains_db(scene, meas["tx_position"], meas["rx_position"], **RT_KW)
    calibration = ln.fit_calibration(meas, predicted, tx_power_dbm=LORA.tx_power_dbm,
                                     cable_loss_db=RADIO.cable_loss_db, bandwidth_hz=LORA.bw_hz, rssi_kind=RSSI_KIND)
    print("Offsets [dB]:", {k: round(v, 2) for k, v in calibration.offset_db.items()})
    print(f"Shadowing sigma {calibration.shadowing_std_db:.2f} dB | fading sigma {calibration.fading_std_db} dB | "
          f"noise figure {calibration.noise_figure_db:.1f} dB")
    print(f"Location-pair RMSE: raw RT {calibration.rmse_uncalibrated_db:.2f} dB -> "
          f"calibrated (leave-one-out) {calibration.rmse_calibrated_loo_db:.2f} dB over {len(calibration.pairs)} pairs")
    gain_used = calibration.apply(gain_rt, link_class)
    radio_used = calibration.radio_model(RADIO)

    lost = ~meas["received"]
    if lost.any():
        cal_gain = calibration.apply(predicted, meas["link_class"])
        tx_p = meas.get("tx_power_dbm", np.full(len(predicted), LORA.tx_power_dbm))
        snr_pred = np.where(np.isfinite(tx_p), tx_p, LORA.tx_power_dbm) - 2 * RADIO.cable_loss_db + cal_gain \
                   - radio_used.noise_dbm(LORA.bw_hz)
        sf_meas = meas.get("sf", np.full(len(predicted), LORA.sf))
        sf_meas = np.where(np.isfinite(sf_meas), sf_meas, LORA.sf)
        ok = np.isfinite(snr_pred)
        offset, width = ln.fit_per_curve(snr_pred[ok], meas["received"][ok], sf_meas[ok].astype(int))
        print(f"PER curve: 50% point {offset:+.2f} dB from the SF limit, width {width:.2f} dB (absorbs fading)")
        radio_used = replace(radio_used, per_snr50_offset_db=offset, per_width_db=width, fading_std_db=0.0)

    pairs = calibration.pairs
    fig, ax = plt.subplots(figsize=(6.5, 5), layout="constrained")
    for cls in sorted({p["link_class"] for p in pairs}):
        res = np.array([p["mean_db"] for p in pairs if p["link_class"] == cls])
        ax.scatter(np.full(len(res), cls), res, alpha=0.7, label=f"{cls} (offset {calibration.offset_db[cls]:+.1f} dB)")
    ax.axhline(0, color="black", lw=0.8)
    ax.set(ylabel="Measured − predicted signal [dB] per location pair", title="Ray-tracer residuals before calibration")
    ax.legend()
    plt.show()
else:
    print("No measurements configured: using uncalibrated ray-traced gains and the assumed RadioModel.")

## 5. Run the network simulation

Each end node generates messages (Poisson or periodic) into a FIFO queue. Per packet and receiver, the simulator decides one outcome:

| Outcome | Meaning |
|---|---|
| `ok` | Detected, no collision and passed the PER draw |
| `not_detected` | SNR below the SF limit minus the detection margin |
| `collision` | Time-weighted interference from overlapping packets violated the SIR threshold (same SF: about 6 dB capture; cross-SF: the rejection matrix in `lora_netsim.SIR_THRESHOLD_DB`) |
| `per_loss` | Detected and collision-free but lost to noise (probability = PER(SNR)) |
| `half_duplex` | The receiver was transmitting (for example, a gateway sending an ACK) |
| `receiver_busy` | All demodulators were locked onto other packets (8 at a gateway, 1 at an SX126x node) |
| `not_listening` / `config_mismatch` | The receiver was asleep, or on another SF/channel |

**Latency** = time from message generation to the end of the first successfully decoded transmission. It includes queueing, duty-cycle waiting, retries and airtime. **PDR** counts unique messages delivered, so duplicates from a lost ACK count once. Reported RSSI/SNR imitate what a radio logs: RSSI includes noise and interference, and SNR is quantised to 0.25 dB.

In [ ]:
def build_nodes(n_end, sf=None, period_s=TRAFFIC_PERIOD_S, traffic=TRAFFIC):
    nodes = [ln.Node(s["name"], lora=LORA, gateway=True) for s in gateway_sites]
    for k in range(n_end):
        lora = replace(LORA, sf=sf or SF_PER_NODE.get(k, LORA.sf), tx_power_dbm=POWER_PER_NODE.get(k, LORA.tx_power_dbm))
        dest = None if TOPOLOGY == "star" else (G + (k + 1) % n_end,)
        nodes.append(ln.Node(pool[k]["name"], lora=lora, dest=dest, period_s=period_s, traffic=traffic,
                             rx_always_on=(TOPOLOGY == "peer")))
    return nodes

def sub_gain(n_end):
    idx = np.r_[0:G, G:G + n_end]
    return gain_used[np.ix_(idx, idx)]

nodes = build_nodes(N_END_NODES)
result = ln.simulate(nodes, sub_gain(N_END_NODES), radio_used, SIM)
summary = ln.summarize(result)
o = summary["overall"]
print(f"Messages {o['generated']}, delivered {o['delivered']}  ->  PDR {o['pdr']:.3f}")
print(f"Latency mean {o['latency_mean_s']*1e3:.0f} ms, median {o['latency_p50_s']*1e3:.0f} ms, p95 {o['latency_p95_s']*1e3:.0f} ms")
print(f"Per-attempt reception {o['attempt_prr']:.3f}, collision fraction {o['collision_fraction']:.3f}, "
      f"channel utilisation {o['channel_utilization']}")
print("Outcomes at intended receivers:", o["outcomes"])

cols = ["node", "sf", "generated", "delivered", "pdr", "attempts_per_msg", "latency_p50_s", "latency_p95_s",
        "rssi_report_dbm_mean", "snr_report_db_mean", "energy_per_delivered_mj"]
lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
for row in summary["per_node"]:
    lines.append("| " + " | ".join(f"{row[c]:.3g}" if isinstance(row[c], float) else str(row[c]) for c in cols) + " |")
display(Markdown("\n".join(lines)))

OUT_DIR = REPO_ROOT / "tutorials" / "rt-campus" / "outputs"
OUT_DIR.mkdir(exist_ok=True)
ln.write_csv(result.messages, OUT_DIR / "messages.csv")
ln.write_csv(result.receptions, OUT_DIR / "receptions.csv")
print("Per-message and per-reception records written to", OUT_DIR)

In [ ]:
per_node = summary["per_node"]
ok = [r for r in result.receptions if r["kind"] == "data" and r["outcome"] == "ok"]
latency = np.sort([m["delivered_s"] - m["gen_s"] for m in result.messages if m["delivered"]])

fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), layout="constrained")
axes[0, 0].bar([r["node"] for r in per_node], [r["pdr"] for r in per_node], color="#0072b2")
axes[0, 0].set(ylim=(0, 1.05), ylabel="PDR", title="Packet delivery ratio per node")
if len(latency):
    axes[0, 1].step(latency * 1e3, np.arange(1, len(latency) + 1) / len(latency), where="post")
    axes[0, 1].set_xscale("log")
axes[0, 1].set(xlabel="Latency [ms]", ylabel="CDF", title="End-to-end latency of delivered messages")
axes[1, 0].hist([r["rssi_report_dbm"] for r in ok], bins=30, color="#009e73")
axes[1, 0].set(xlabel="Reported RSSI [dBm]", ylabel="Packets", title="RSSI of received packets")
axes[1, 1].hist([r["snr_report_db"] for r in ok], bins=30, color="#d55e00")
axes[1, 1].axvline(ln.SNR_LIMIT_DB[LORA.sf], color="black", ls="--", label=f"SF{LORA.sf} SNR limit")
axes[1, 1].set(xlabel="Reported SNR [dB]", ylabel="Packets", title="SNR of received packets")
axes[1, 1].legend()
for ax in axes.flat:
    ax.grid(alpha=0.2)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 7), layout="constrained")
pdr_values = [r["pdr"] for r in per_node]
for ax, floor in zip(axes, floors):
    mappable = draw_floor(ax, floor, values=pdr_values)
fig.colorbar(mappable, ax=axes, shrink=0.6, label="PDR")
fig.suptitle("Where packets are lost: node colour = PDR")
plt.show()

## 6. Inspect one node's packets

The per-reception records keep every attempt with its outcome, received power, SNR and number of interferers. Use them to check *why* a node underperforms before changing settings. Weak propagation (`not_detected` / `per_loss`) calls for a different SF, TX power or position. Contention (`collision`, `receiver_busy`) calls for less traffic, more channels or scheduling.

In [ ]:
worst = min(per_node, key=lambda r: r["pdr"])
idx = [n.name for n in nodes].index(worst["node"])
recs = [r for r in result.receptions if r["src"] == idx and r["kind"] == "data"]
from collections import Counter
print(f"{worst['node']}: PDR {worst['pdr']:.3f}; outcomes {dict(Counter(r['outcome'] for r in recs))}")
print(f"Mean received power {np.mean([r['prx_dbm'] for r in recs]):.1f} dBm vs SF{nodes[idx].lora.sf} sensitivity "
      f"{radio_used.sensitivity_dbm(nodes[idx].lora.sf, LORA.bw_hz):.1f} dBm")

## 7. Sweep node count × spreading factor

All nodes share one SF here, and each point averages several seeds. Higher SF buys sensitivity but multiplies airtime (SF12 ≈ 26× SF7 for this payload), so at a fixed traffic rate it eventually loses to collisions. Gains come from the traced pool, so no extra ray tracing is needed.

In [ ]:
sweep = {}
for sf in SWEEP_SFS:
    for n_end in SWEEP_NODE_COUNTS:
        runs = [ln.summarize(ln.simulate(build_nodes(n_end, sf=sf, period_s=SWEEP_PERIOD_S), sub_gain(n_end),
                                         radio_used, replace(SIM, seed=s)))["overall"] for s in SWEEP_SEEDS]
        sweep[sf, n_end] = {k: np.nanmean([r[k] for r in runs])
                            for k in ("pdr", "latency_p95_s", "collision_fraction", "attempt_prr")}
    print(f"SF{sf} done")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), layout="constrained")
for sf in SWEEP_SFS:
    x = SWEEP_NODE_COUNTS
    axes[0].plot(x, [sweep[sf, n]["pdr"] for n in x], "o-", label=f"SF{sf}")
    axes[1].plot(x, [sweep[sf, n]["latency_p95_s"] for n in x], "o-", label=f"SF{sf}")
    axes[2].plot(x, [sweep[sf, n]["collision_fraction"] for n in x], "o-", label=f"SF{sf}")
axes[0].set(ylabel="PDR", ylim=(0, 1.05))
axes[1].set(ylabel="p95 latency [s]", yscale="log")
axes[2].set(ylabel="Fraction of attempts lost to collisions")
for ax in axes:
    ax.set(xlabel="End nodes")
    ax.grid(alpha=0.2)
    ax.legend()
fig.suptitle(f"{TOPOLOGY} topology, one message per {SWEEP_PERIOD_S:g} s per node, "
             f"{'confirmed' if SIM.confirmed else 'unconfirmed'}, {len(SWEEP_SEEDS)} seeds")
plt.show()

## 8. Interpreting the results

- **Propagation** is a deterministic ray-traced mean plus statistical shadowing and fading. Without section 4 it inherits the provisional material proxies, so treat absolute RSSI as ±10 dB until calibrated. Relative comparisons (SF A vs B, position X vs Y) are more robust than absolute PDR.
- **Receiver model.** A logistic PER waterfall anchored at Semtech's SNR limits; literature SIR thresholds for capture and inter-SF rejection; preamble lock-on and a fixed number of demodulators. It is not a bit-exact SX1262. Replace the PER curve via section 4 and the SIR matrix with measured collision tests if collisions dominate.
- **Not modelled:** adjacent-channel leakage, other campus 923 MHz users, moving people/nodes, clock drift, LoRaWAN ADR/join/network server and the RX2 window. Energy counts only TX and ACK-window RX with assumed currents.
- **Good practice:** report metrics with the seed spread, keep `messages.csv`/`receptions.csv` with the configuration, and validate against a held-out set of measured locations, not those used for calibration.